# End of week 1 exercise

To demonstrate your familiarity with your AI API choice (Z.AI), OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

Note that GLM models (from Z.AI) are used in this assessment

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [3]:
# constants

MODEL_GPT = 'glm-5'
MODEL_LLAMA = 'qwen-2.5-coder'

In [4]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('ZAI_API_KEY')

# Z.AI client for GLM models
client = OpenAI(
    api_key=api_key,
    base_url="https://api.z.ai/api/paas/v4/"
)

# Ollama client for local open-source models
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [5]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [6]:
# Get glm-5 to answer, with streaming

system_prompt = "You are a helpful technical assistant that explains code clearly and concisely."

stream = client.chat.completions.create(
    model=MODEL_GPT,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ],
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response), display_id=display_handle.display_id)

This line of code creates a **generator** that yields a list of **unique author names** from a collection of books, ignoring any books that do not have an author listed.

Here is the step-by-step breakdown:

### 1. The Set Comprehension (`{...}`)
The code inside the curly braces `{}` is a **set comprehension**.
*   **Iteration:** It loops through every `book` in the `books` iterable (usually a list of dictionaries).
*   **Filtering:** The `if book.get("author")` part checks if the "author" key exists and has a value. It skips entries where the author is `None` or missing.
*   **Extraction:** It retrieves the value of the "author" key.
*   **Deduplication:** Because it creates a **set**, all duplicate author names are automatically removed. If "J.K. Rowling" appears five times in the data, she will appear only once in the set.

### 2. The Delegation (`yield from`)
*   `yield from` is Python syntax used inside a generator function.
*   It takes an iterable (in this case, the set of unique authors) and yields each item from that set individually to the caller.

### Summary of "Why"
This pattern is used when you need a **stream of unique values** without loading the entire result into memory at once (aside from the temporary set of unique names).

It effectively combines three operations into one line:
1.  **Filtering** out missing data.
2.  **Deduplicating** the results.
3.  **Lazily yielding** the data.

In [7]:
# Get qwen-2.5-coder to answer (running locally via Ollama)

response = ollama_client.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
)

display(Markdown(response.choices[0].message.content))

This code snippet uses the `yield from` statement to yield each unique author found in a list of dictionaries called `books`. It filters out any books that do not have an author by checking if the author is not `None` using the `if` condition.
In summary, this code is used to extract unique authors from a list of dictionaries and yield them one by one. 
The reason for using `yield from` is to simplify the code and make it more readable. Instead of manually iterating over the set and yielding each element using a `for` loop, `yield from` allows us to yield all elements in the set in a single statement.

Here's an example use case:
```python
books = [
    {"title": "Book One", "author": "John Smith"},
    {"title": "Book Two", "author": "David Johnson"},
    {"title": "Book Three", "author": "John Smith"},
    {"title": "Book Four", "author": None},
]

unique_authors = (author for author in get_unique_authors(books))

for author in unique_authors:
    print(author)
```
In this example, we define a list of dictionaries called `books` with some books that have an author and some that don't. We then create a generator expression `(author for author in get_unique_authors(books))` to extract unique authors from the `books` list. Finally, we loop through the generator expression to print each unique author. 
The output of this code would be:
```
John Smith
David Johnson
```